# U.S. Chronic Disease Indicators — CRISP-DM Walkthrough

This notebook runs the pipeline stage by stage: **PySpark cleaning → EDA → modeling**.

Make sure the raw CDI CSV is at `data/raw/U_S__Chronic_Disease_Indicators.csv` before running (see the README).

> Run from the project root, or the relative paths in `config.yaml` won't resolve.

In [ ]:
import os
# Ensure we're at the project root so `import src...` and config paths work.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("cwd:", os.getcwd())

from src.utils import load_config
cfg = load_config()
cfg["target_topics"]

## Stage 1 — Data preparation (PySpark)

Applies the seven cleaning steps and writes the curated dataset. Expect **309,215 × 34 → 69,272 × 13**.

In [ ]:
from src import data_preparation
data_preparation.run(cfg)

### Peek at the curated data

In [ ]:
import pandas as pd
df = pd.read_csv(cfg["paths"]["cleaned_csv"])
print(df.shape)
df.head()

## Stage 2 — Exploratory data analysis

Writes figures to `reports/figures/`.

In [ ]:
from src import eda
eda.run(cfg)

In [ ]:
from IPython.display import Image, display
import glob
for path in sorted(glob.glob(cfg["paths"]["figures_dir"] + "/0[1-4]*.png")):
    display(Image(path))

## Stage 3 — Modeling & evaluation

Trains Linear Regression, Random Forest, and XGBoost (baseline + tuned). Watch R² jump from the linear baseline to the tree models.

In [ ]:
from src import modeling
metrics = modeling.run(cfg)
metrics

In [ ]:
from IPython.display import Image
Image(cfg["paths"]["figures_dir"] + "/05_model_comparison.png")

## Takeaway

Label-encoded categoricals leave the linear model near R² ≈ 0, while tree-based models exploit the non-linear interactions between location, topic, question, and demographics to reach R² ≈ 0.78–0.80. XGBoost is the best predictor of chronic-disease burden here.